In [ ]:
from jax import config as jax_config
jax_config.update("jax_enable_x64", True)

import matplotlib.pyplot as plt

import colors
import helpers_builders
import plot_funcs
import importlib
import numpy as np
from config import CFG
from EquilibriumClass import EquilibriumClass
from StateClass import StateClass
from SupervisorClass import SupervisorClass
from VariablesClass import VariablesClass

In [ ]:
colors_lst, red, custom_cmap = colors.color_scheme()
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=colors_lst)

## Configured model

All user-editable parameters are in `config.py`.

In [ ]:
import config
importlib.reload(config)
from config import CFG

import VariablesClass
importlib.reload(VariablesClass)
from VariablesClass import VariablesClass

import SupervisorClass
importlib.reload(SupervisorClass)
from SupervisorClass import SupervisorClass

import EquilibriumClass
importlib.reload(EquilibriumClass)
from EquilibriumClass import EquilibriumClass

import StateClass
importlib.reload(StateClass)
from StateClass import StateClass

Variabs = VariablesClass(CFG, plot_potential=CFG.Output.plot_potential)
Sprvsr = SupervisorClass(CFG)
State = StateClass(Variabs)
State.get_free_dof_parameters(Variabs)
Eq = EquilibriumClass(Variabs)

## Input pulse

In [ ]:
impulse_data = Sprvsr.program_impulse()
plot_funcs.plot_impulse(Sprvsr.timepoints, impulse_data)

## Simulations

In [ ]:
importlib.reload(plot_funcs)
importlib.reload(helpers_builders)

# empty tuple for simulating same pulse, different initial phase
solutions_by_phase = {}

# loop initial phase
for initial_phase in Sprvsr.initial_phases:
    State.get_current_local_global_states(Variabs, initial_phase)
    sol_free = Eq.solve(State, Sprvsr)
    sol_global, F_global = State.reshape_local_to_global(Variabs, Sprvsr, sol_free)
    solutions_by_phase[initial_phase] = sol_global
    if CFG.Output.plot_responses:
        fig, axes = plot_funcs.plot_response(sol_global, F_global, Sprvsr.timepoints, sup_title=f"Initial phase {initial_phase}")
        transform_fig, transform_axes = plot_funcs.plot_laplace_fourier(F_global, Sprvsr.timepoints, sup_title=f"Initial phase {initial_phase}", laplace_sigma=1)

## State and endpoint-force comparison

In [ ]:
importlib.reload(plot_funcs)

for initial_phase, sol_global in solutions_by_phase.items():
    final_relative_displacement = (sol_global[-1, 2, 1:-1] - sol_global[-1, 0, 1:-1])
    final_state = State.get_system_state(final_relative_displacement, threshold=CFG.Output.state_threshold)
    print(f"Initial {initial_phase} -> final state: {final_state} " f"({helpers_builders.state_to_number(final_state)})")

if CFG.Output.compare_endpoint_forces and len(solutions_by_phase) == 2:
    _, _, force_delay = plot_funcs.plot_force_comparison(solutions_by_phase, Sprvsr.timepoints, Variabs.k1, Sprvsr.start_time, 
                                                         CFG.Output.force_arrival_threshold_fraction, show_transform=True)
    print(f"Force-arrival delay: {force_delay * 1e3:.1f} ms")

## Laplace and FFT for all eight initial states
Rows contain 000; 001, 010, 100; 011, 101, 110; and 111. Run the simulations for all eight phases first.

In [ ]:
state_rows = (("000",), ("001", "010", "100"), ("011", "101", "110"), ("111",))
missing_phases = [phase for row in state_rows for phase in row if phase not in solutions_by_phase]
if missing_phases:
    raise ValueError(f"Run the simulations with all eight initial_phases in config.py first. Missing: {', '.join(missing_phases)}")

laplace_sigma = 0.0  # s^-1, matching the individual-state plots
laplace_states_fig, laplace_states_axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True, sharey=True, layout="constrained")
fft_states_fig, fft_states_axes = plt.subplots(4, 3, figsize=(15, 13), sharex=True, sharey=True, layout="constrained")
laplace_states_fig.suptitle(fr"Endpoint Force Laplace — all initial states ($\sigma={laplace_sigma:g}$ s$^{{-1}}$)", fontsize=16)
fft_states_fig.suptitle("Endpoint Force FFT — all initial states", fontsize=16)
for axes in (laplace_states_axes, fft_states_axes):
    for row in (0, 3):
        for column in (0, 2):
            axes[row, column].set_visible(False)
for row_index, phases in enumerate(state_rows):
    for column_index, phase in zip((1,) if len(phases) == 1 else range(3), phases):
        laplace_ax, fft_ax = laplace_states_axes[row_index, column_index], fft_states_axes[row_index, column_index]
        solution = np.asarray(solutions_by_phase[phase])
        forces = Variabs.k1 * (solution[:, 0, :-1] - solution[:, 0, 1:])
        for force, color, label in zip((forces[:, 0], forces[:, -1]), colors_lst[:2], ("First truss", "Final truss")):
            frequencies, spectrum = helpers_builders.force_fft(Sprvsr.timepoints, force)
            nonnegative = frequencies >= 0
            frequencies, spectrum = frequencies[nonnegative], spectrum[nonnegative]
            _, laplace = helpers_builders.force_laplace(Sprvsr.timepoints, force, laplace_sigma + 2j * np.pi * frequencies)
            laplace_ax.plot(frequencies, np.abs(laplace), color=color, label=label)
            fft_ax.plot(frequencies, spectrum.real, color=color, linestyle="-", label=f"{label} (real)")
            fft_ax.plot(frequencies, spectrum.imag, color=color, linestyle=":", label=f"{label} (imaginary)")
        laplace_ax.set_ylabel("Transform magnitude (N s)")
        fft_ax.set_ylabel("Transform (N s)")
        for ax in (laplace_ax, fft_ax):
            ax.set_title(phase, fontsize=14)
            ax.set_xlabel("Frequency (Hz)")
            ax.set_xlim(0, float(frequencies[-1]) / 2 if frequencies[-1] > 0 else 1.0)
            ax.tick_params(labelbottom=True, labelleft=True)
            ax.legend(fontsize=8)
            ax.grid(False)
plt.show()

## Amplitude study

In [ ]:
# Quick amplitude study: every initial buckle configuration, one sine cycle at 23.3 Hz
from copy import copy
from matplotlib.colors import BoundaryNorm, ListedColormap

A_values_mm = np.linspace(-7.0, 7.0, 60)  # 1 mm steps, including zero
f_values_hz = np.array([60.0])
study_phases = tuple(format(number, f'0{Variabs.n_physical_units}b') for number in range(2 ** Variabs.n_physical_units))
study_supervisor = copy(Sprvsr)
study_supervisor.timepoints = Sprvsr.timepoints
study_state = StateClass(Variabs)
study_state.get_free_dof_parameters(Variabs)
study_eq = EquilibriumClass(Variabs)

def simulate_amplitude_final_state(A_mm: float, f_hz: float) -> str:
    """Return the final state after one sine cycle from the initialized study state."""
    study_supervisor.program_impulse(impulse_type='single_sine_cycle', amplitude=A_mm * 1e-3, frequency=f_hz)
    solution = study_eq.solve(study_state, study_supervisor)
    final_displacement = np.asarray(solution[-1, 0, 1::2] - solution[-1, 0, 0::2])
    if not np.all(np.isfinite(final_displacement)):
        raise RuntimeError(f'Nonfinite final displacement at A={A_mm:g} mm, f={f_hz:g} Hz.')
    return study_state.get_system_state(final_displacement, threshold=CFG.Output.state_threshold)

parameter_states_by_phase, state_number_grids_by_phase = {}, {}
for study_phase in study_phases:
    study_state.get_current_local_global_states(Variabs, study_phase)
    parameter_states = [(float(A_mm), float(f_hz), simulate_amplitude_final_state(float(A_mm), float(f_hz))) for A_mm in A_values_mm for f_hz in f_values_hz]
    parameter_states_by_phase[study_phase] = parameter_states
    state_number_grids_by_phase[study_phase] = np.array([helpers_builders.state_to_number(row[2]) for row in parameter_states]).reshape(len(A_values_mm), len(f_values_hz))
    print(f'Initial {study_phase}: completed {len(parameter_states)} amplitude simulations')

In [ ]:
# Single-frequency final-state maps, sharing the same discrete colors for all initial phases
state_labels = list(study_phases)
_, _, study_cmap = colors.color_scheme()
state_cmap = ListedColormap(study_cmap(np.linspace(0, 1, len(state_labels))))
state_norm = BoundaryNorm(np.arange(-0.5, len(state_labels) + 0.5), state_cmap.N)
A_edges_mm = np.concatenate(([A_values_mm[0] - (A_values_mm[1] - A_values_mm[0]) / 2], (A_values_mm[:-1] + A_values_mm[1:]) / 2, [A_values_mm[-1] + (A_values_mm[-1] - A_values_mm[-2]) / 2]))
f_edges_hz = f_values_hz[0] + np.array([-0.5, 0.5])  # Display width only; just one simulated frequency
study_ncols = min(4, len(study_phases))
study_nrows = (len(study_phases) + study_ncols - 1) // study_ncols
amplitude_fig, amplitude_axes = plt.subplots(study_nrows, study_ncols, figsize=(3 * study_ncols, 4 * study_nrows), sharex=True, sharey=True, squeeze=False, layout='constrained')
for ax, study_phase in zip(amplitude_axes.flat, study_phases):
    state_map = ax.pcolormesh(f_edges_hz, A_edges_mm, state_number_grids_by_phase[study_phase], cmap=state_cmap, norm=state_norm, shading='flat')
    ax.set(xlabel='Frequency (Hz)', ylabel='Amplitude (mm)', title=f'Initial {study_phase}', xticks=f_values_hz, yticks=np.arange(np.ceil(A_values_mm[0]), np.floor(A_values_mm[-1]) + 1, 1.0), ylim=(A_edges_mm[0], A_edges_mm[-1]))
    ax.tick_params(labelbottom=True, labelleft=True)
    ax.grid(False)
for ax in list(amplitude_axes.flat)[len(study_phases):]:
    ax.set_visible(False)
state_colorbar = amplitude_fig.colorbar(state_map, ax=list(amplitude_axes.flat)[:len(study_phases)], ticks=range(len(state_labels)), label='Final state', pad=0.02)
state_colorbar.ax.set_yticklabels(state_labels)
amplitude_fig.suptitle(f'Final state after one sine-cycle input at {f_values_hz[0]} Hz', fontsize=16)
plt.show()